<a href="https://colab.research.google.com/github/ericryhr/MAI-OR/blob/master/Deliverable2/marti_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deliverable 2: model APCNet


# MMSeg installation

In [5]:
# Check nvcc version
!nvcc -V
# Check GCC version
!gcc --version
# Check Python version
!python --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Python 3.11.11


In [6]:
!pip --quiet install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121

In [8]:
!pip --quiet install -U openmim

In [9]:
!mim install mmcv==2.1.0


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/local/bin/mim", line 8, in <module>
    sys.exit(cli())
  File "/usr/local/lib/python3.11/dist-packages/click/core.py", line 1161, in __call__
    return self.main(*args, **kwargs)
  File "/usr/local/lib/python3.11/dist-packages/click/core.py", line 1082, in main
    rv = self.invoke(ctx)
  File "/usr/local/lib/python3.11/dist-packages/click/core.py", line 1697, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
  File "/usr/local/lib/python3.11/dist-packages/click/core.py", line 1443, in in

In [10]:
!rm -rf mmsegmentation
!git clone -b main https://github.com/open-mmlab/mmsegmentation.git
%cd mmsegmentation
!pip install -e .

Cloning into 'mmsegmentation'...
remote: Enumerating objects: 16493, done.
remote: Total 16493 (delta 0), reused 0 (delta 0), pack-reused 16493 (from 1)
Receiving objects: 100% (16493/16493), 44.23 MiB | 25.01 MiB/s, done.
Resolving deltas: 100% (11495/11495), done.
/content/mmsegmentation/mmsegmentation
Obtaining file:///content/mmsegmentation/mmsegmentation
  Preparing metadata (setup.py) ... done
  Attempting uninstall: mmsegmentation
    Found existing installation: mmsegmentation 1.2.2
    Uninstalling mmsegmentation-1.2.2:
      Successfully uninstalled mmsegmentation-1.2.2
  Running setup.py develop for mmsegmentation


In [11]:
# Check Pytorch installation
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

2.1.0+cu121 False


In [12]:
# Check MMSegmentation installation
import mmseg
print(mmseg.__version__)

1.2.2


# Downloading Dataset

In [ ]:
import os
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:
#!brew install axel -qq     #MacOS
!apt-get install axel       #Linux

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  axel
0 upgraded, 1 newly installed, 0 to remove and 29 not upgraded.
Need to get 58.7 kB of archives.
After this operation, 204 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 axel amd64 2.17.11-1 [58.7 kB]
Fetched 58.7 kB in 1s (51.5 kB/s)
Selecting previously unselected package axel.
(Reading database ... 126209 files and directories currently installed.)
Preparing to unpack .../axel_2.17.11-1_amd64.deb ...
Unpacking axel (2.17.11-1) ...
Setting up axel (2.17.11-1) ...
Processing triggers for man-db (2.10.2-1) ...


Images

In [ ]:
!axel https://s3.amazonaws.com/ifashionist-dataset/images/train2020.zip  -n 7 --quiet # training images
!axel https://s3.amazonaws.com/ifashionist-dataset/images/val_test2020.zip -n 7 --quiet # validation images

In [ ]:
os.makedirs('dataset', exist_ok=True)
!unzip train2020.zip -d dataset
!unzip val_test2020.zip -d dataset

Streaming output truncated to the last 5000 lines.
  inflating: dataset/train/ab3336508bbcceea810d6984312ed24d.jpg  
  inflating: dataset/train/8f32dea17754957b8e019cfa940dbe03.jpg  
  inflating: dataset/train/4b2c88d0d6beb0f34e21422717ca6357.jpg  
  inflating: dataset/train/7ea9648135a54293902f5bdcb905d55c.jpg  
  inflating: dataset/train/133ccc1d8bb7a2b5dcf739ba0d6dc755.jpg  
  inflating: dataset/train/21685d9dcadc17a3fb6ea2386fc313bf.jpg  
  inflating: dataset/train/05adb75d6a1dbdb1c70475ea8c5812a8.jpg  
  inflating: dataset/train/c5dacae7ac188f76b0da5aa60f65cafb.jpg  
  inflating: dataset/train/2a7be033a7dbaa0e9b98ad4d724e6c9b.jpg  
  inflating: dataset/train/43582be804d81524d6b4c9def0b885a9.jpg  
  inflating: dataset/train/574dda9c4663c2c0ba5379d718b68b46.jpg  
  inflating: dataset/train/aa44ea197382653e78127af7e0d44a21.jpg  
  inflating: dataset/train/cc38a7b8ce7fe53bd3c020072d5914c8.jpg  
  inflating: dataset/train/91193b67f675c0052ae43d1d675713fa.jpg  
  inflating: dataset/trai

Annotations

In [ ]:
!axel https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_train2020.json -n 7 --quiet -o dataset/instances_attributes_train2020.json
!axel https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_val2020.json -n 7 --quiet -o dataset/instances_attributes_val2020.json
!axel https://s3.amazonaws.com/ifashionist-dataset/annotations/info_test2020.json -n 7 --quiet -o dataset/info_test2020.json

FORMATS:
```
category{
  "id" : int,
  "name" : str,
  "supercategory" : str,  # parent of this label
  "level": int,           # levels in the taxonomy
  "taxonomy_id": string,
}

annotation{
  "id" : int,
  "image_id" : int,
  "category_id" : int,
  "attribute_ids": [int],
  "segmentation" : [polygon] or [rle]
  "bbox" : [x,y,width,height], # int
  "area" : int
  "iscrowd": int (1 or 0)
}

image{
  "id" : int,
  "width" : int,
  "height" : int,
  "file_name" : str,
  "license" : int,
  "time_captured": string,
  "original_url": string,
  "isstatic": int, 0: the original_url is not a static url,
  "kaggle_id": str,
}
```

In [ ]:
with open("dataset/instances_attributes_train2020.json", "r") as file:
    data = json.load(file)